In [11]:
# Lab: Running Local BLAST with Bio.Blast.Applications

import sys

# Run this in Google Colab
!{sys.executable} -m pip install biopython

import os
import subprocess

from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord

print("=" * 70)
print("LAB 1: RUNNING LOCAL BLAST WITH BIO.BLAST.APPLICATIONS")
print("=" * 70)

# ===== PART 1: Installing BLAST in Colab =====
print("\n" + "=" * 70)
print("PART 1: Installing BLAST Tools in Colab")
print("=" * 70)

print("\nInstalling BLAST+ package...")
os.system("apt-get update -qq")
os.system("apt-get install -y ncbi-blast+ > /dev/null 2>&1")

# Verify installation
result = os.system("blastn -version > /dev/null 2>&1")
if result == 0:
    print("✓ BLAST+ successfully installed")
else:
    print("✗ BLAST+ installation failed")

# Check BLAST version
v = subprocess.run("blastn -version", shell=True, capture_output=True, text=True).stdout
print("\nBLASTn version: ", v )
v = subprocess.run("blastx -version", shell=True, capture_output=True, text=True).stdout
print("\nBLASTx version: ", v )
v = subprocess.run("tblastx -version", shell=True, capture_output=True, text=True).stdout
print("\ntBLASTx version: ", v )

# ===== PART 2: Creating Sample Sequences =====
print("\n" + "=" * 70)
print("PART 2: Creating Sample Sequences and Database - continued in next cell")
print("=" * 70)

LAB 1: RUNNING LOCAL BLAST WITH BIO.BLAST.APPLICATIONS

PART 1: Installing BLAST Tools in Colab

Installing BLAST+ package...
✓ BLAST+ successfully installed

BLASTn version:  blastn: 2.12.0+
 Package: blast 2.12.0, build Mar  8 2022 16:19:08


BLASTx version:  blastx: 2.12.0+
 Package: blast 2.12.0, build Mar  8 2022 16:19:08


tBLASTx version:  tblastx: 2.12.0+
 Package: blast 2.12.0, build Mar  8 2022 16:19:08


PART 2: Creating Sample Sequences and Database - continued in next cell


In [2]:
%%shell
# This cell is for running Unix/Linux shell commands only

# Create a nucleotide query file
echo ">query_seq" > query.fasta
echo "ATGGCGGTACTAGTAACTGTAGACTGTGATGGTGTGAGTGCCTAG" >> query.fasta

# Create a dummy PROTEIN database (for blastx)
echo ">prot_1" > protein_db.fasta
echo "MAVLVSVTVDCDCDEVAL" >> protein_db.fasta
echo ">prot_2" >> protein_db.fasta
echo "MKTLLILALCIGTVWG" >> protein_db.fasta

# Create a dummy NUCLEOTIDE database (for tblastx)
echo ">nucl_1" > nucleotide_db.fasta
echo "ATGGCGGTACTAGTAACTGTAGACTGTGAT" >> nucleotide_db.fasta

# Format databases using NCBI BLAST+ tools
makeblastdb -in protein_db.fasta -dbtype prot -out local_prot_db
makeblastdb -in nucleotide_db.fasta -dbtype nucl -out local_nucl_db



Building a new DB, current time: 07/04/2026 13:19:20
New DB name:   /content/local_prot_db
New DB title:  protein_db.fasta
Sequence type: Protein
Keep MBits: T
Maximum file size: 1000000000B
Adding sequences from FASTA; added 2 sequences in 0.000648975 seconds.




Building a new DB, current time: 07/04/2026 13:19:20
New DB name:   /content/local_nucl_db
New DB title:  nucleotide_db.fasta
Sequence type: Nucleotide
Keep MBits: T
Maximum file size: 1000000000B
Adding sequences from FASTA; added 1 sequences in 0.000275135 seconds.




In [3]:
# Back to Part 2, now that we set up the sequences

print("1. Running local BLASTX (Nucleotide query vs Protein DB)...")
# Setup blastx command (outfmt=5 means XML output)
blast_command = [
    "blastx",
    "-query",  "query.fasta",
    "-db",     "local_prot_db",
    "-out",    "blastx_results.xml",
    "-outfmt", "5",
    "-evalue", "10"
]
# Execute the command
result = subprocess.run(blast_command, capture_output=True, text=True, check=True)

print("\n2. Running local TBLASTX (Nucleotide query vs Nucleotide DB)...")
# Setup tblastx command
tblastx_command = [
    "tblastx",
    "-query",  "query.fasta",
    "-db",     "local_nucl_db",
    "-out",    "tblastx_results.xml",
    "-outfmt", "5",
    "-evalue", "10"
]
# Execute the command
result = subprocess.run(tblastx_command, capture_output=True, text=True, check=True)

# Verify the files were created
print("\nGenerated Files:")
print([f for f in os.listdir() if f.endswith(".xml")])

1. Running local BLASTX (Nucleotide query vs Protein DB)...

2. Running local TBLASTX (Nucleotide query vs Nucleotide DB)...

Generated Files:
['tblastx_results.xml', 'blastx_results.xml']


In [4]:

# ===== PART 3: Send a Sequence to the BLAST server =====
from Bio.Blast import NCBIWWW
from Bio import SeqIO
import time

print("\n" + "=" * 70)
print("PART 3: Remote BLAST run")
print("=" * 70)

# 1. Define an unknown nucleotide sequence (Example: part of the human beta-globin gene)
unknown_seq = "CTCGAGTGAGAGAACGATATGTAACTGTAGACTGTGATGGTGTGAGTGCCTAG"

print("Submitting query to NCBI... (This may take 1-3 minutes)")
start_time = time.time()

# 2. Run the remote BLAST search
# Parameters: program ("blastn"), database ("nt"), sequence
result_handle = NCBIWWW.qblast("blastn", "nt", unknown_seq)

# 3. Save the results to an XML file for later parsing
with open("remote_blast_results.xml", "w") as out_file:
    out_file.write(result_handle.read())

result_handle.close()
print(f"Search complete in {round(time.time() - start_time, 2)} seconds. Results saved to 'remote_blast_results.xml'.")

# 4. Open the XML file generated in Lab 1
result_handle = open("remote_blast_results.xml")

# 5. Parse the XML data
from Bio.Blast import NCBIXML
blast_record = NCBIXML.read(result_handle)

# 6. Set a strict E-value threshold
E_VALUE_THRESH = 0.01

print(f"Showing alignments with E-value < {E_VALUE_THRESH}:\n")

# 7. Iterate through alignments and their respective HSPs
for alignment in blast_record.alignments:
    for hsp in alignment.hsps:
        if hsp.expect < E_VALUE_THRESH:
            print(f"*** Alignment: {alignment.title[:50]}... ***")
            print(f"Length: {alignment.length}")
            print(f"E-value: {hsp.expect}")
            print(f"Bit Score: {hsp.bits}")
            # Show the actual alignment visual (first 50 characters)
            print(f"Query: {hsp.query[:50]}...")
            print(f"Match: {hsp.match[:50]}...")
            print(f"Subjt: {hsp.sbjct[:50]}...")
            print("-" * 50)

result_handle.close()
print("\n" + "=" * 70)
print("PART 3: Remote BLAST run done")
print("=" * 70)




PART 3: Remote BLAST run
Submitting query to NCBI... (This may take 1-3 minutes)
Search complete in 60.96 seconds. Results saved to 'remote_blast_results.xml'.
Showing alignments with E-value < 0.01:


PART 3: Remote BLAST run done
